In [ ]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 972.7 kB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.41.1
    Uninstalling openai-2.41.1:
      Successfully uninstalled openai-2.41.1


In [ ]:
import openai
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset
from google import genai
from google.genai import types

In [ ]:
openai.api_key = 'your-api'
# comment the below line if you use original OPENAI api
openai.api_base = 'https://api.avalai.org/v1'
login("your-key")

In [ ]:
dataset = load_dataset("cais/mmlu",'college_mathematics')

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

college_mathematics/test-00000-of-00001.(…):   0%|          | 0.00/16.6k [00:00<?, ?B/s]

college_mathematics/validation-00000-of-(…):   0%|          | 0.00/5.00k [00:00<?, ?B/s]

college_mathematics/dev-00000-of-00001.p(…):   0%|          | 0.00/5.16k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
def get_AI_answer(prompt):
    response = openai.ChatCompletion.create(
        model="gemma-4-26b-a4b-it",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature = 1.0
    )
    result = response['choices'][0]['message']['content'].strip().lower()
    return result

In [ ]:
prompt_COT_fa = f'''
به پرسش زیر قدم به قدم فکر کنید و زنجیره افکار (chain of thought) خود برای رسیدن به پاسخ را به طور کامل شرح دهید و تنها یک گزینه را انتخاب کنید.

## پرسش
{{question}}

## گزینه ها
[A] : {{option1}}
[B] : {{option2}}
[C] : {{option3}}
[D] : {{option4}}
'''

prompt_COT_en = f'''
Think about the following question step-by-step, explain your chain of thought for reaching the answer in detail, and select only one option.

## Question
{{question}}

## Options
[A] : {{option1}}
[B] : {{option2}}
[C] : {{option3}}
[D] : {{option4}}
'''

In [ ]:
def append_record_to_excel(file_path, Question,
                           correct_answer, model_prompt, AI_answer):
    new_record = {
        'Question': Question,
        'correct_answer': correct_answer,
        'model_prompt':  model_prompt,
        'AI_answer': AI_answer
    }
    new_record_df = pd.DataFrame([new_record])
    try:
        existing_df = pd.read_excel(file_path)
        updated_df = pd.concat([existing_df, new_record_df], ignore_index=True)
    except FileNotFoundError:
        updated_df = new_record_df

    updated_df.to_excel(file_path, index=False)

def convert_to_letters(number):
    if number == 0:
        return 'A'
    elif number == 1:
        return 'B'
    elif number == 2:
        return 'C'
    elif number == 3:
        return 'D'
    else:
        return None

In [ ]:
EXP_NUM = 1
counter = 0
for example in dataset["test"]:
    prmpt = prompt_COT_en.format(question=example["question"], option1=example["choices"][0], option2=example["choices"][1], option3=example["choices"][2], option4=example["choices"][3])

    AI_answer = get_AI_answer(prmpt)
    file_path = 'gemma-4-26b-a4b-it_college_mathematics_En_no'+str(EXP_NUM)+'.xlsx'
    append_record_to_excel(file_path = file_path,Question=example["question"],
                           correct_answer=convert_to_letters(example["answer"]),
                           model_prompt = prmpt, AI_answer=AI_answer)
    counter += 1
    print("Question number " + str(counter) + " has been solved.")
    print('================================================================')